# Projekt 2 — MLP w PyTorch (Wine Quality)

Uzupełnienie do notebooka scikit-learn: ta sama baza **`WineQT.csv`**, ale sieć neuronowa zaimplementowana w **PyTorch** (warstwy `Linear`, `ReLU`, `CrossEntropyLoss`).

**Wymaganie:** `pip install torch` (w aktywnym venv projektu).

## Plan
1. Przygotowanie danych (ten sam podział 80/20, stratyfikacja).
2. Definicja i trening MLP.
3. Wykres funkcji straty w epokach.
4. Macierz pomyłek i metryki na zbiorze testowym.
5. Porównanie z wynikami sklearn z `results/`.


In [ ]:
# Jeśli brak PyTorch: odkomentuj następną linię w terminalu i uruchom komórkę ponownie
# !pip install torch

%matplotlib inline

import warnings
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
except ImportError as e:
    raise ImportError("Zainstaluj PyTorch: pip install torch") from e

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    ConfusionMatrixDisplay,
    classification_report,
)

ROOT = Path.cwd()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import DATA_PATH, FEATURE_COLUMNS, TARGET_COLUMN, RANDOM_STATE, TEST_SIZE, RESULTS_DIR
from src.experiment import wczytaj_pelna_ramke, podziel_na_cechy_i_etykiete

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Urządzenie:", DEVICE)


## 1. Dane treningowe i testowe

- **X** — 11 cech chemicznych (bez `Id`).
- **y** — ocena `quality`; w PyTorch mapujemy na indeksy 0…K−1.
- **StandardScaler** dopasowany **tylko na train** (jak w pipeline sklearn).


In [ ]:
df = wczytaj_pelna_ramke(DATA_PATH)
X, y, _ = podziel_na_cechy_i_etykiete(df)

classes = sorted(np.unique(y))
class_to_idx = {c: i for i, c in enumerate(classes)}
y_idx = np.array([class_to_idx[v] for v in y])

X_train, X_test, y_train, y_test = train_test_split(
    X, y_idx, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_idx
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

n_features = X_train_s.shape[1]
n_classes = len(classes)
print(f"Train: {len(y_train)}, Test: {len(y_test)}, Cechy: {n_features}, Klasy: {classes}")


## 2. Architektura sieci MLP

Prosta sieć feed-forward: **128 → 64 → K klas**, aktywacja ReLU, dropout 0,2 po pierwszej warstwie ukrytej (ograniczenie przeuczenia).


In [ ]:
class WineMLP(nn.Module):
    def __init__(self, n_in: int, n_out: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, n_out),
        )

    def forward(self, x):
        return self.net(x)


model = WineMLP(n_features, n_classes).to(DEVICE)
print(model)


## 3. Trening (40 epok)

Używamy **CrossEntropyLoss** (wieloklasowa klasyfikacja) i optymizatora **Adam**.  
Zapisujemy stratę w każdej epoce — wykres pokaże, czy model się stabilizuje.


In [ ]:
BATCH_SIZE = 64
EPOCHS = 40
LR = 1e-3

train_ds = TensorDataset(
    torch.tensor(X_train_s, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long),
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

opt = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss()

historia_loss = []

model.train()
for ep in range(EPOCHS):
    ep_loss = 0.0
    n = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        opt.step()
        ep_loss += loss.item() * len(yb)
        n += len(yb)
    historia_loss.append(ep_loss / n)
    if (ep + 1) % 10 == 0:
        print(f"Epoka {ep+1}/{EPOCHS}, loss={historia_loss[-1]:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, EPOCHS + 1), historia_loss, marker="o", markersize=3)
ax.set_xlabel("Epoka")
ax.set_ylabel("Cross-entropy (średnia na batch)")
ax.set_title("Krzywa uczenia — MLP PyTorch")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Ewaluacja na zbiorze testowym (20%)


In [ ]:
model.eval()
with torch.no_grad():
    logits = model(torch.tensor(X_test_s, dtype=torch.float32).to(DEVICE))
    y_pred = logits.argmax(1).cpu().numpy()

acc = accuracy_score(y_test, y_pred)
ba = balanced_accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

print(f"Accuracy:          {acc:.4f}")
print(f"Balanced accuracy: {ba:.4f}")
print(f"F1-macro:          {f1:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=[str(c) for c in classes], zero_division=0))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=[str(c) for c in classes]
).plot(ax=ax, cmap="Purples", colorbar=True)
ax.set_title("Macierz pomyłek — MLP PyTorch (test 20%)")
plt.tight_layout()
plt.show()


## 5. Porównanie z MLP (scikit-learn) z pliku wyników

Jeśli wcześniej uruchomiono `python run_experiment.py`, można porównać PyTorch z najlepszym wariantem **sklearn MLP** na tym samym typie podziału (oba: 80/20, seed 42, skalowanie).


In [ ]:
test_csv = RESULTS_DIR / "wyniki_test_mlp.csv"
if test_csv.is_file():
    sk = pd.read_csv(test_csv).iloc[0]
    print("sklearn MLP (z results/wyniki_test_mlp.csv):")
    print(f"  model: {sk['model']}")
    print(f"  accuracy_test: {sk['accuracy_test']:.4f}")
    print(f"  balanced_accuracy_test: {sk['balanced_accuracy_test']:.4f}")
    print(f"  f1_macro_test: {sk['f1_macro_test']:.4f}")
    print()
    print("PyTorch MLP (ten notebook):")
    print(f"  accuracy_test: {acc:.4f}")
    print(f"  balanced_accuracy_test: {ba:.4f}")
    print(f"  f1_macro_test: {f1:.4f}")
else:
    print("Brak results/wyniki_test_mlp.csv — uruchom: python run_experiment.py")


## 6. Wnioski (PyTorch)

- Implementacja **PyTorch** pozwala śledzić **krzywą straty** i swobodnie modyfikować architekturę.
- Wyniki są **porównywalne rzędu wielkości** ze sklearn MLP — różnice wynikają z innej inicjalizacji, braku early stopping i innej implementacji optimizerów.
- Przy **niezbalansowanych** klasach warto rozważyć `weight` w `CrossEntropyLoss` lub metryki z wagami klas.
- Do sprawozdania główne wyniki bierz z notebooka **scikit-learn** i `src/experiment.py` (oficjalny pipeline projektu).
